
# Appendix 02: Circular Tables And Charts

Source span: printed pages 353-380; PDF pages 366-393. The PDF span was inspected for table families, chart purposes, and notation only. This notebook does not copy printed tables, long text, or chart images. It replaces the static appendix tables with executable calculators, simulations, and visual checks.

## Chapter Goal

The printed appendix collects circular lookup tables: von Mises quantiles, the Bessel-ratio function `A_1`, inverse concentration estimates, uniformity critical values, confidence charts for concentration, and multi-sample or rank-style test thresholds. A standalone notebook should not ask the reader to memorize rows. It should show how the rows are made, which domains are safe, and how a fresh critical value can be computed when a sample size or confidence level is not printed.

The guiding question is: what does a table become when it is executable? In this notebook it becomes four linked tools. First, a von Mises quantile and `A_1` dashboard shows how concentration changes angular probability and mean resultant length. Second, a simulation lab estimates critical values for common circular uniformity diagnostics. Third, a confidence-chart replacement uses repeated samples to display uncertainty in the concentration estimate. Fourth, an interactive surface makes the dependence on sample size and significance level visible.

The calculations are deliberately small enough to audit. They are not meant to reproduce every printed row. Instead they teach the computational contract behind the tables: state the statistic, state the null or model, simulate or solve under that model, and save the residuals that show the table replacement is internally consistent.



## Visualization Storyboard And Library Routing

The source span contains many table families, so the notebook groups them by computational job rather than by page order.

- **Von Mises distribution tables.** Quantiles, `A_1(kappa)`, and `A_1^{-1}(Rbar)` are deterministic calculators. Matplotlib is the right durable medium because the reader needs labeled curves and a compact inversion view. SciPy supplies `vonmises.ppf`, Bessel functions, and root finding.

- **Uniformity critical-value tables.** Rayleigh, Watson, Kuiper, circular range, and spacing diagnostics are better understood through null simulations than through isolated numbers. The visual target is the distribution of a statistic under uniformly distributed angles and the way critical values change with sample size.

- **Concentration confidence charts.** The printed charts are replaced by a parametric simulation map: for each true concentration, sample from a von Mises model, estimate concentration from `Rbar`, and plot empirical quantile bands. This gives the reader the same practical information as a chart while exposing Monte Carlo uncertainty and boundary behavior.

- **Interactive critical surface.** Plotly is used for a single interactive HTML artifact where interaction matters: the upper critical value as a function of sample size and alpha. This helps the reader see interpolation risk between rows and columns.

Every artifact is concept-named and book-local. The saved JSON file stores monotonicity checks, inverse round-trip residuals, simulation quantile ordering, and artifact existence records.


In [ ]:

from pathlib import Path
import sys


def find_book_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (
            (candidate / "AGENTS.md").exists()
            and (candidate / "scripts" / "validate_dirstats_course.py").exists()
            and (candidate / "utils").exists()
        ):
            return candidate
    raise RuntimeError("Could not locate Directional-Statistics course root")


BOOK_ROOT = find_book_root(Path.cwd())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = "appendix-02"
source_span = {"printed": "353-380", "pdf": "366-393"}
print(f"Course root: {BOOK_ROOT}")
source_span


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy import optimize, special, stats

from utils.artifacts import display_artifact, save_json, save_matplotlib, save_plotly_html
from utils.validation import assert_artifacts

rng = np.random.default_rng(2402)
np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def wrap_pi(theta):
    return (theta + np.pi) % (2.0 * np.pi) - np.pi


def A1(kappa):
    kappa = np.asarray(kappa, dtype=float)
    return special.ive(1, kappa) / special.ive(0, kappa)


def A1_inverse(resultant: float) -> float:
    if resultant <= 1e-12:
        return 0.0
    if not 0.0 < resultant < 1.0:
        raise ValueError("resultant must lie in (0, 1)")
    upper = max(8.0, 1.0 / max(1e-4, 2.0 * (1.0 - resultant)))
    f = lambda value: A1(value) - resultant
    while f(upper) < 0:
        upper *= 2.0
    return float(optimize.brentq(f, 1e-10, upper, maxiter=100))


def resultant_length(theta, axis=-1):
    c = np.mean(np.cos(theta), axis=axis)
    s = np.mean(np.sin(theta), axis=axis)
    return np.sqrt(c * c + s * s)


def rayleigh_stat(theta):
    n = theta.shape[-1]
    rbar = resultant_length(theta, axis=-1)
    return 2.0 * n * rbar * rbar


def watson_u2_stat(theta):
    theta = np.sort((theta % (2.0 * np.pi)) / (2.0 * np.pi), axis=-1)
    n = theta.shape[-1]
    i = np.arange(1, n + 1)
    centered = theta - (2.0 * i - 1.0) / (2.0 * n)
    return np.sum(centered * centered, axis=-1) + 1.0 / (12.0 * n) - n * np.mean(centered, axis=-1) ** 2


def kuiper_stat(theta):
    theta = np.sort((theta % (2.0 * np.pi)) / (2.0 * np.pi), axis=-1)
    n = theta.shape[-1]
    i = np.arange(1, n + 1)
    d_plus = np.max(i / n - theta, axis=-1)
    d_minus = np.max(theta - (i - 1.0) / n, axis=-1)
    return d_plus + d_minus


def circular_range_degrees(theta):
    u = np.sort(theta % (2.0 * np.pi), axis=-1)
    gaps = np.diff(np.concatenate([u, u[..., :1] + 2.0 * np.pi], axis=-1), axis=-1)
    return np.degrees(2.0 * np.pi - np.max(gaps, axis=-1))


def min_spacing_degrees(theta):
    u = np.sort(theta % (2.0 * np.pi), axis=-1)
    gaps = np.diff(np.concatenate([u, u[..., :1] + 2.0 * np.pi], axis=-1), axis=-1)
    return np.degrees(np.min(gaps, axis=-1))


def simulate_uniform_statistics(n_values, reps=5000):
    rows = []
    for n in n_values:
        theta = rng.uniform(-np.pi, np.pi, size=(reps, n))
        rows.append({
            "n": int(n),
            "rayleigh": rayleigh_stat(theta),
            "watson_u2": watson_u2_stat(theta),
            "kuiper": kuiper_stat(theta),
            "range_degrees": circular_range_degrees(theta),
            "min_spacing_degrees": min_spacing_degrees(theta),
        })
    return rows



## Von Mises Quantiles, `A_1`, And Inverse Lookup

Several circular tables reduce to deterministic functions. The von Mises quantile table asks: for a centered circular model with concentration `kappa`, what angular interval contains a chosen probability? The `A_1` table asks: what mean resultant length does that concentration imply? The inverse table asks the estimation question in reverse: given `Rbar`, what concentration estimate solves `A_1(kappa)=Rbar`?

The dashboard below keeps these three questions together. The quantile fan uses degrees because the appendix tables are meant for human lookup. The `A_1` panel shows monotonicity. The inverse panel makes the boundary visible: as `Rbar` approaches one, the solved concentration grows rapidly. The final panel shows sample density curves, reminding the reader that these tables are all consequences of one circular probability model.


In [ ]:

kappa_grid = np.linspace(0.001, 12.0, 450)
kappa_table = np.array([0.0, 0.5, 1.0, 2.0, 4.0, 8.0])
prob_levels = np.array([0.50, 0.80, 0.90, 0.95])

fig, axes = plt.subplots(2, 2, figsize=(12, 8.5))
ax = axes[0, 0]
for q in prob_levels:
    half_width = []
    for kappa in kappa_table:
        if kappa == 0:
            half_width.append(180.0 * q)
        else:
            half_width.append(np.degrees(stats.vonmises.ppf((1.0 + q) / 2.0, kappa)))
    ax.plot(kappa_table, half_width, marker="o", label=f"central {int(q*100)}%")
ax.set_title("Von Mises central quantile fan")
ax.set_xlabel("concentration kappa")
ax.set_ylabel("half width in degrees")
ax.legend()

ax = axes[0, 1]
ax.plot(kappa_grid, A1(kappa_grid), color="#345995")
ax.set_title("A_1(kappa) = I_1(kappa) / I_0(kappa)")
ax.set_xlabel("concentration kappa")
ax.set_ylabel("mean resultant length")
ax.set_ylim(0, 1.02)

ax = axes[1, 0]
r_grid = np.linspace(0.05, 0.97, 28)
inv_grid = np.array([A1_inverse(r) for r in r_grid])
ax.plot(r_grid, inv_grid, marker="o", color="#E07A5F")
ax.set_title("Executable inverse table A_1^{-1}(Rbar)")
ax.set_xlabel("observed Rbar")
ax.set_ylabel("estimated kappa")

ax = axes[1, 1]
theta = np.linspace(-np.pi, np.pi, 720)
for kappa in [0.0, 0.5, 1.5, 4.0]:
    density = np.full_like(theta, 1.0 / (2.0 * np.pi)) if kappa == 0 else stats.vonmises.pdf(theta, kappa)
    ax.plot(np.degrees(theta), density, label=f"kappa={kappa:g}")
ax.set_title("Centered von Mises density family")
ax.set_xlabel("angle in degrees")
ax.set_ylabel("density")
ax.legend()

fig.suptitle("Appendix 02 deterministic circular table replacements", y=1.02, fontsize=14)
fig.tight_layout()
vm_path = save_matplotlib(fig, TOPIC, "von-mises-tables", "von-mises-quantile-and-inversion-dashboard.png")
plt.close(fig)
display_artifact(vm_path, width=920)

vm_diagnostics = {
    "A1_monotone": bool(np.all(np.diff(A1(kappa_grid)) > 0)),
    "A1_inverse_roundtrip_max_abs_error": float(max(abs(A1(A1_inverse(r)) - r) for r in r_grid)),
    "quantile_half_widths_decrease_with_kappa": bool(
        all(np.all(np.diff([np.degrees(stats.vonmises.ppf((1.0 + q) / 2.0, k)) if k > 0 else 180.0 * q for k in kappa_table]) <= 0) for q in prob_levels)
    ),
}
vm_diagnostics



## Uniformity Critical Values By Simulation

Uniformity tables are not just numbers; each row is a model-based promise. If the angles are independent and uniformly distributed, then a statistic such as Rayleigh's `2 n Rbar^2`, Watson's `U^2`, Kuiper's statistic, circular range, or minimum spacing has a reference distribution. A critical-value table stores upper or lower quantiles of that reference distribution.

The simulation below builds a small version of such a table. It deliberately includes statistics with different visual behavior. Rayleigh responds to a preferred direction. Watson and Kuiper compare circular empirical distribution functions without choosing a permanent cut point. Circular range and spacing ask about empty arcs and clustering gaps. Seeing them side by side helps the reader choose a diagnostic rather than treating all uniformity tables as interchangeable.


In [ ]:

n_values = [8, 12, 20, 40]
uniform_rows = simulate_uniform_statistics(n_values, reps=7000)
alpha_levels = [0.10, 0.05, 0.01]

fig, axes = plt.subplots(2, 2, figsize=(12, 8.5))
stat_specs = [
    ("rayleigh", "Rayleigh 2 n Rbar^2", "upper"),
    ("watson_u2", "Watson U^2", "upper"),
    ("kuiper", "Kuiper V", "upper"),
    ("range_degrees", "Circular range", "lower"),
]
critical_table = []
for ax, (key, title, tail) in zip(axes.ravel(), stat_specs):
    for alpha in alpha_levels:
        values = []
        for row in uniform_rows:
            q = alpha if tail == "lower" else 1.0 - alpha
            val = float(np.quantile(row[key], q))
            values.append(val)
            critical_table.append({"statistic": key, "tail": tail, "n": row["n"], "alpha": alpha, "critical_value": val})
        ax.plot(n_values, values, marker="o", label=f"alpha={alpha:g}")
    ax.set_title(title)
    ax.set_xlabel("sample size n")
    ax.set_ylabel("critical value")
    ax.legend()

fig.suptitle("Monte Carlo replacements for circular uniformity critical-value tables", y=1.02, fontsize=14)
fig.tight_layout()
uniform_path = save_matplotlib(fig, TOPIC, "uniformity-critical-values", "uniformity-critical-value-simulator.png")
plt.close(fig)
display_artifact(uniform_path, width=920)

# Check quantile ordering for each statistic and sample size.
ordering_ok = True
for key, _, tail in stat_specs:
    for n in n_values:
        vals = [item["critical_value"] for item in critical_table if item["statistic"] == key and item["n"] == n]
        if tail == "upper":
            ordering_ok = ordering_ok and (vals[0] < vals[1] < vals[2])
        else:
            ordering_ok = ordering_ok and (vals[0] > vals[1] > vals[2])

uniform_diagnostics = {
    "uniform_simulation_reps_per_n": 7000,
    "critical_quantile_ordering_ok": bool(ordering_ok),
    "rayleigh_mean_near_asymptotic_two": float(abs(np.mean(uniform_rows[-1]["rayleigh"]) - 2.0)),
    "critical_table_rows": len(critical_table),
}
uniform_diagnostics



## Concentration Confidence Chart Replacement

The source appendix includes charts for concentration intervals. The computational version asks a direct question: if the true concentration is `kappa`, how variable is the estimate obtained by solving `A_1(kappa_hat)=Rbar`? Repeating that experiment gives a band for the estimator.

This is a chart, not a formal replacement for every exact interval in the literature. Its teaching value is that it exposes the shape of the problem. Near zero concentration, many samples have small resultants, so the estimator is compressed against the boundary. At high concentration, the estimator remains positive but can become skewed because `A_1^{-1}` is steep near one. The width of the band shrinks with sample size, as a confidence chart should.


In [ ]:

def simulate_kappa_hat(true_kappa, n, reps=2500):
    samples = stats.vonmises.rvs(true_kappa, size=(reps, n), random_state=rng)
    rbar = resultant_length(samples, axis=1)
    return np.array([A1_inverse(min(float(r), 0.999999)) for r in rbar])

true_kappas = np.array([0.0, 0.5, 1.0, 2.0, 4.0, 8.0])
fig, ax = plt.subplots(figsize=(10.5, 5.7))
confidence_rows = []
for n, color in [(15, "#E07A5F"), (40, "#345995")]:
    lows = []
    meds = []
    highs = []
    for kappa in true_kappas:
        hats = simulate_kappa_hat(kappa, n, reps=2200)
        lo, med, hi = np.quantile(hats, [0.05, 0.50, 0.95])
        lows.append(lo); meds.append(med); highs.append(hi)
        confidence_rows.append({"n": n, "true_kappa": float(kappa), "q05": float(lo), "q50": float(med), "q95": float(hi)})
    ax.plot(true_kappas, meds, marker="o", color=color, label=f"median estimate, n={n}")
    ax.fill_between(true_kappas, lows, highs, color=color, alpha=0.18, label=f"90% simulation band, n={n}")
ax.plot(true_kappas, true_kappas, color="black", linestyle="--", linewidth=1.2, label="ideal y=x")
ax.set_title("Executable concentration confidence chart")
ax.set_xlabel("true concentration kappa")
ax.set_ylabel("estimated concentration from A_1^{-1}(Rbar)")
ax.legend()
fig.tight_layout()
confidence_path = save_matplotlib(fig, TOPIC, "concentration-confidence", "kappa-confidence-chart-replacement.png")
plt.close(fig)
display_artifact(confidence_path, width=900)

band_width_by_n = {
    n: float(np.mean([row["q95"] - row["q05"] for row in confidence_rows if row["n"] == n and row["true_kappa"] > 0]))
    for n in [15, 40]
}
confidence_diagnostics = {
    "confidence_rows": len(confidence_rows),
    "bands_shrink_with_larger_n": bool(band_width_by_n[40] < band_width_by_n[15]),
    "all_quantile_rows_ordered": bool(all(row["q05"] <= row["q50"] <= row["q95"] for row in confidence_rows)),
    "mean_band_width_n15": band_width_by_n[15],
    "mean_band_width_n40": band_width_by_n[40],
}
confidence_diagnostics



## Interactive Critical Surface

A printed critical-value table has rows and columns. Interpolation between them can be opaque. The interactive surface below shows upper Rayleigh critical values as a function of sample size and significance level. It is intentionally limited to one statistic because the purpose is not to bury the reader in surfaces; the purpose is to make the table geometry inspectable.

Read the surface in two directions. Along fixed alpha, the finite-sample critical value stabilizes as `n` grows. Along fixed sample size, demanding a smaller alpha raises the threshold. This is exactly what a critical-value table encodes, but the surface makes the monotone directions visible.


In [ ]:

surface_n = np.array([8, 10, 12, 16, 20, 30, 40, 60])
surface_alpha = np.array([0.20, 0.10, 0.05, 0.025, 0.01])
surface_rows = simulate_uniform_statistics(surface_n, reps=5000)
Z = np.array([
    [float(np.quantile(row["rayleigh"], 1.0 - alpha)) for row in surface_rows]
    for alpha in surface_alpha
])
N, A = np.meshgrid(surface_n, surface_alpha)
critical_fig = go.Figure(data=[go.Surface(x=N, y=A, z=Z, colorscale="Viridis")])
critical_fig.update_layout(
    title="Rayleigh critical-value surface under circular uniformity",
    scene=dict(
        xaxis_title="sample size n",
        yaxis_title="alpha",
        zaxis_title="upper critical value",
        yaxis=dict(autorange="reversed"),
    ),
    margin=dict(l=0, r=0, t=50, b=0),
    height=620,
)
surface_path = save_plotly_html(critical_fig, TOPIC, "interactive", "rayleigh-critical-value-surface.html", include_plotlyjs=True)
display_artifact(surface_path, width="100%", height=640)

surface_diagnostics = {
    "surface_shape": list(Z.shape),
    "critical_values_positive": bool(np.all(Z > 0)),
    "critical_values_increase_as_alpha_decreases": bool(np.all(np.diff(Z, axis=0) > 0)),
}
surface_diagnostics



## How To Use These Table Replacements

A table replacement should be used with the same care as a printed table. First identify the model or null distribution. For the von Mises quantile fan, the model is a centered von Mises distribution. For the uniformity lab, the null is independent uniform angles. For the confidence chart, the data are generated from a von Mises model and the estimator is the inverse Bessel-ratio estimator. Changing any of those assumptions changes the table.

Second, inspect the domain. The inverse concentration calculator accepts `0 < Rbar < 1`. A sample with `Rbar` very close to one is not just a big number; it is a boundary case where the inverse curve is steep. Uniformity critical values depend on sample size and on whether the test uses an upper or lower tail. Confidence bands depend on the simulation design.

Third, keep a record of the computation. The JSON file saved by this notebook stores the simulation size, monotonicity checks, quantile ordering checks, and artifact records. That makes the appendix auditable. A future chapter can reuse the same pattern: define the statistic, simulate or solve the reference distribution, and preserve enough diagnostics that the lookup can be trusted.


In [ ]:

numeric_checks = {
    **vm_diagnostics,
    **uniform_diagnostics,
    **confidence_diagnostics,
    **surface_diagnostics,
    "source_span": source_span,
    "critical_table_preview": critical_table[:8],
    "confidence_table": confidence_rows,
    "all_boolean_checks_pass": bool(
        all(value for value in vm_diagnostics.values() if isinstance(value, bool))
        and all(value for value in uniform_diagnostics.values() if isinstance(value, bool))
        and all(value for value in confidence_diagnostics.values() if isinstance(value, bool))
        and all(value for value in surface_diagnostics.values() if isinstance(value, bool))
    ),
}
assert numeric_checks["all_boolean_checks_pass"], numeric_checks
assert numeric_checks["A1_inverse_roundtrip_max_abs_error"] < 1e-9
assert numeric_checks["rayleigh_mean_near_asymptotic_two"] < 0.12
checks_path = save_json(numeric_checks, TOPIC, "checks", "circular-table-replacement-checks.json")
display_artifact(checks_path)
numeric_checks



## Takeaways

Appendix 02 is best read as a catalog of circular calculators.

- Von Mises quantiles, `A_1(kappa)`, and `A_1^{-1}(Rbar)` are deterministic functions. They should be plotted and inverted directly when a new value is needed.
- Uniformity critical values are model-based simulation targets. The statistic and tail direction must be named before a number is meaningful.
- Confidence charts for concentration are uncertainty maps for an estimator, not decorative curves. Their width and skewness reveal where circular concentration estimation is fragile.
- Multi-row printed tables can be replaced by small reproducible simulations when the notebook records seeds, sample sizes, quantile ordering, and residual checks.

The larger course can now cite this appendix as an executable reference. When a later chapter asks for a critical value or a concentration lookup, the reader should know which calculator to run, what geometry it respects, and what sanity check confirms the result.


In [ ]:

final_sanity = {
    "source_span": source_span,
    "artifacts": assert_artifacts(
        [vm_path, uniform_path, confidence_path, surface_path, checks_path],
        min_bytes=100,
    ),
    "core_checks": {
        "A1_monotone": vm_diagnostics["A1_monotone"],
        "A1_inverse_roundtrip_under_tolerance": vm_diagnostics["A1_inverse_roundtrip_max_abs_error"] < 1e-9,
        "uniform_critical_quantiles_ordered": uniform_diagnostics["critical_quantile_ordering_ok"],
        "confidence_bands_shrink_with_sample_size": confidence_diagnostics["bands_shrink_with_larger_n"],
        "rayleigh_surface_monotone_in_alpha": surface_diagnostics["critical_values_increase_as_alpha_decreases"],
    },
    "pdf_used_for": "source orientation only; no copied tables, charts, screenshots, or page crops",
    "standalone_contract": "executable table replacements, concept-named artifacts, simulation diagnostics, and invariant checks",
}
assert all(final_sanity["core_checks"].values()), final_sanity
final_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json")
assert final_path.exists() and final_path.stat().st_size > 20
final_sanity
